In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):

    def __init__(self, head_size, d_model, block_size):
        super().__init__()
        self.key   = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # [B, T, head_size]
        q = self.query(x)  # [B, T, head_size]
        v = self.value(x)  # [B, T, head_size]

        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)  # [B, T, T]
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)  # [B, T, T]

        out = weights @ v  # [B, T, head_size]
        return out

In [3]:
B = 4
T = 8
d_model = 65
head_size = 16

head = Head(head_size=head_size, d_model=d_model, block_size=T)

x = torch.randn(B, T, d_model)
out = head(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

Input shape:  torch.Size([4, 8, 65])
Output shape: torch.Size([4, 8, 16])


In [4]:
# Run a forward pass and capture the weights
# Temporarily modify Head to return weights too, or just redo the computation:

with torch.no_grad():
    k = head.key(x)
    q = head.query(x)
    scores = q @ k.transpose(-2, -1) * (head_size ** -0.5)
    scores = scores.masked_fill(head.tril[:T, :T] == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)

print("Attention weights for batch item 0:")
print(weights[0].round(decimals=2))


Attention weights for batch item 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4600, 0.5400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3700, 0.4500, 0.1800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1800, 0.1900, 0.3700, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1100, 0.2200, 0.3000, 0.1400, 0.2200, 0.0000, 0.0000, 0.0000],
        [0.1400, 0.1900, 0.1700, 0.1400, 0.2100, 0.1600, 0.0000, 0.0000],
        [0.1500, 0.1300, 0.1600, 0.1300, 0.1700, 0.1100, 0.1500, 0.0000],
        [0.2000, 0.1400, 0.0500, 0.1200, 0.2000, 0.0700, 0.0900, 0.1300]])
